# SeaData End-to-End Pipeline Walkthrough

This notebook demonstrates the full SeaData workflow: data loading, training, calibrated evaluation, and visual reporting.

In [ ]:
import sys
import os
from pathlib import Path
import pandas as pd

# Set working directory to project root
if Path.cwd().name == "notebooks":
    os.chdir("..")

project_root = str(Path.cwd())
if project_root not in sys.path:
    sys.path.append(project_root)

from src.cli import build_experiment, load_raw_frame
from src.explain.report import generate_visual_report
from src.utils.io import read_json

# 1. Load Configuration
config_path = "configs/mlp_water_level.yaml"
config, experiment = build_experiment(config_path)

print(f"Experiment initialized: {experiment.name()}")
print(f"Working directory: {Path.cwd()}")

In [ ]:
# 2. Run Full Pipeline (Preprocess -> Train -> Evaluate -> Visualize)
frame = load_raw_frame(config)
experiment.run(frame)

print("Pipeline execution complete!")

In [ ]:
# 3. Inspect Metrics
eval_path = config.paths.evaluation_json
metrics = read_json(eval_path)

print(f"Accuracy: {metrics.get('accuracy', 0):.2%}")
print(f"Event Recall: {metrics.get('event_recall', 0):.2%}")
if "mean_event_confidence" in metrics:
    print(f"Avg Event Confidence: {metrics.get('mean_event_confidence', 0):.1%}")

In [ ]:
# 4. Generate Premium Visual Report
report_dir = config.paths.evaluation_json.parent
importance_csv = report_dir / "feature_importance.csv"

if importance_csv.exists():
    importance_df = pd.read_csv(importance_csv)

    generate_visual_report(
        importance_df=importance_df,
        metrics=metrics,
        output_path=report_dir / "notebook_visual_report.pdf",
        experiment_name=experiment.name(),
        plots_dir=report_dir,
    )
    print(f"PDF Report generated: {report_dir}/notebook_visual_report.pdf")
else:
    print("Run SHAP analysis first to generate feature importance CSV.")